# Reuters News Topic Classification - Model Training

This notebook implements and trains different models for the Reuters news topic classification task. We'll compare several approaches:

1. Multinomial Naive Bayes with TF-IDF
2. Linear SVM with TF-IDF (unigrams)
3. Linear SVM with TF-IDF (bigrams)
4. MiniLM embeddings with Logistic Regression
5. RAG-kMajority
6. RAG-CentroidNN
7. RAG-LLM

## Setup and Data Loading

In [4]:
import logging
import warnings
import random
import os
import numpy as np
from src.dataset import load_data
from src.algorithms.naive_bayes import NaiveBayesClassifier
from src.algorithms.linear_svm import LinearSVMClassifier, LinearSVMBigrams
from src.algorithms.transformer_logreg import TransformerLogReg
from src.rag import load_kmajority, load_centroid, load_llm
from src.rag.adapter_sklearn import RagSklearnAdapter
from src.evaluation import run_evaluations

# Disable HuggingFace tokenizers parallelism
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Load dataset
N_CLASSES = 10
X_train, y_train, X_test, y_test, label_names = load_data(N_CLASSES)
print(f'Train docs: {len(X_train):,},  Test docs: {len(X_test):,}')
print('Labels:', label_names)

Train docs: 6,337,  Test docs: 2,477
Labels: ['earn', 'acq', 'crude', 'interest', 'money-fx', 'trade', 'grain', 'corn', 'dlr', 'money-supply']


## Initialize and Train All Models

In [5]:
# Initialize all models
models = {
    'Naive Bayes': NaiveBayesClassifier(),
    'Linear SVM': LinearSVMClassifier(),
    'TF-IDF bigrams + SVM': LinearSVMBigrams(),
    'MiniLM + LogReg': TransformerLogReg(),
    'RAG-kMajority': RagSklearnAdapter(load_kmajority(top_k=5)),
    'RAG-CentroidNN': RagSklearnAdapter(load_centroid()),
    'RAG-LLM': RagSklearnAdapter(load_llm(top_k=5, model="gpt-4o-mini"))
}


models = {
    'Naive Bayes': NaiveBayesClassifier(),
    'Linear SVM': LinearSVMClassifier(),
}


INFO | Use pytorch device_name: cpu
INFO | Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO | Loading faiss with AVX2 support.
INFO | Successfully loaded faiss with AVX2 support.
INFO | Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.


OPENAI_API_KEY loaded successfully.


In [6]:
# Train and evaluate all models
results, significance_test = run_evaluations(
    models=models,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    label_names=label_names
)

# Print results for each model
for name, metrics in results.items():
    print(f"\n{name} Results:")
    print(f"Accuracy: {metrics['accuracy']:.3f}")
    print(f"Macro F1: {metrics['f1_macro']:.3f}")

▶ Training Naive Bayes … done  ✓
▶ Training Linear SVM … done  ✓

All files saved under /home/marcmaceira/projects/reuters-rag-classifier/results
Significance test: Linear SVM vs Naive Bayes → p = 0.0000

Naive Bayes Results:
Accuracy: 0.928
Macro F1: 0.822

Linear SVM Results:
Accuracy: 0.947
Macro F1: 0.867


## Model Comparison

In [8]:
import pandas as pd

df_results = pd.DataFrame(results).T[['accuracy', 'macro_f1', 'weighted_f1']]
df_results.columns = ['Accuracy', 'Macro F1', 'Weighted F1']
df_results = df_results.round(3)

print("Model Comparison:")
print(df_results)

KeyError: "['macro_f1', 'weighted_f1'] not in index"

## Save model and metrics

In [ ]:
# Save all models
import joblib
import os
from src.utils.model_storage import save_model

# Save all models
for name, model in models.items():
    save_model(
        model=model,
        model_name=name,
        metrics=results[name],
        n_classes=N_CLASSES
    )

## Next Steps

In the next notebook, we'll:
1. Perform detailed error analysis
2. Create a confusion matrix to understand model mistakes
3. Implement a semantic search demo using the transformer embeddings